In [1]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("topGlobal")   # cambia si tu carpeta se llama distinto
OUT_DIR = DATA_DIR / "junio_primera_semana"
OUT_DIR.mkdir(exist_ok=True)

date_pat = re.compile(r"(\d{4}-\d{2}-\d{2})")

def first_june_week_df(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Timestamp]:
    # extraer fecha del source_file
    df = df.copy()
    df["chart_date"] = df["source_file"].astype(str).str.extract(date_pat)[0]
    df["chart_date"] = pd.to_datetime(df["chart_date"], errors="coerce")

    # filtrar junio
    june = df[df["chart_date"].dt.month == 6].copy()
    if june.empty:
        return june, None

    # "primera semana de junio" = menor fecha dentro de junio que exista en tus datos
    first_date = june["chart_date"].min()
    out = june[june["chart_date"] == first_date].copy()
    return out, first_date

# procesa todos los globalYYYY.csv
for fp in sorted(DATA_DIR.glob("global*.csv")):
    df = pd.read_csv(fp)

    out, d = first_june_week_df(df)
    if d is None:
        print(f"[WARN] {fp.name}: no hay semanas en junio")
        continue

    out_fp = OUT_DIR / f"{fp.stem}_first_week_june_{d.date()}.csv"
    out.to_csv(out_fp, index=False)
    print(f"[OK] {fp.name}: primera semana de junio = {d.date()} | filas: {len(out)} -> {out_fp.name}")

[WARN] global2016.csv: no hay semanas en junio
[OK] global2017.csv: primera semana de junio = 2017-06-01 | filas: 200 -> global2017_first_week_june_2017-06-01.csv
[OK] global2018.csv: primera semana de junio = 2018-06-07 | filas: 200 -> global2018_first_week_june_2018-06-07.csv
[OK] global2019.csv: primera semana de junio = 2019-06-06 | filas: 200 -> global2019_first_week_june_2019-06-06.csv
[OK] global2020.csv: primera semana de junio = 2020-06-04 | filas: 200 -> global2020_first_week_june_2020-06-04.csv
[OK] global2021.csv: primera semana de junio = 2021-06-03 | filas: 200 -> global2021_first_week_june_2021-06-03.csv
[OK] global2022.csv: primera semana de junio = 2022-06-02 | filas: 200 -> global2022_first_week_june_2022-06-02.csv
[OK] global2023.csv: primera semana de junio = 2023-06-01 | filas: 200 -> global2023_first_week_june_2023-06-01.csv
[OK] global2024.csv: primera semana de junio = 2024-06-06 | filas: 200 -> global2024_first_week_june_2024-06-06.csv
[OK] global2025.csv: prim

In [2]:
import pandas as pd
import requests
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from pathlib import Path
from tqdm import tqdm

# -----------------------------
# Spotify credentials
# -----------------------------

CLIENT_ID = "2a1f45380fda4f0ba4661f6e44882f59"
CLIENT_SECRET = "024f5dc449d04b27aaf01f4da41c95ff"

auth = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

sp = spotipy.Spotify(auth_manager=auth)

# -----------------------------
# archivos a usar
# -----------------------------

files = [
    "topGlobal/junio_primera_semana/global2017_first_week_june_2017-06-01.csv",
    "topGlobal/junio_primera_semana/global2021_first_week_june_2021-06-03.csv",
    "topGlobal/junio_primera_semana/global2025_first_week_june_2025-06-05.csv"
]

# carpeta de salida
OUT = Path("sounds/previews")
OUT.mkdir(parents=True, exist_ok=True)

# -----------------------------
# función para sacar track_id
# -----------------------------

def get_track_id(uri):
    return uri.split(":")[-1]

# -----------------------------
# descargar previews
# -----------------------------

for file in files:

    df = pd.read_csv(file)

    year = file.split("global")[1][:4]

    year_folder = OUT / year
    year_folder.mkdir(exist_ok=True)

    print("Procesando", year)

    for _, row in tqdm(df.iterrows(), total=len(df)):

        track_id = get_track_id(row["uri"])

        try:

            track = sp.track(track_id)
            preview = track["preview_url"]

            if preview is None:
                continue

            audio = requests.get(preview)

            filename = year_folder / f"{track_id}.mp3"

            with open(filename, "wb") as f:
                f.write(audio.content)

        except:
            continue

Procesando 2017


100%|██████████| 200/200 [00:37<00:00,  5.39it/s]


Procesando 2021


100%|██████████| 200/200 [00:33<00:00,  6.03it/s]


Procesando 2025


100%|██████████| 200/200 [00:30<00:00,  6.63it/s]


In [3]:
import pandas as pd
import requests
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from pathlib import Path
from tqdm import tqdm
import re
import time

CLIENT_ID = "2a1f45380fda4f0ba4661f6e44882f59"
CLIENT_SECRET = "024f5dc449d04b27aaf01f4da41c95ff"

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
)

files = [
    "topGlobal/junio_primera_semana/global2017_first_week_june_2017-06-01.csv",
    "topGlobal/junio_primera_semana/global2021_first_week_june_2021-06-03.csv",
    "topGlobal/junio_primera_semana/global2025_first_week_june_2025-06-05.csv"
]

OUT = Path("sounds/previews")
OUT.mkdir(parents=True, exist_ok=True)

def track_id_from_uri(uri: str) -> str:
    return str(uri).split(":")[-1]

def safe_name(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-\.]+", "_", s)   # reemplaza espacios/raros por _
    return s[:120]                      # limita longitud

for file in files:
    df = pd.read_csv(file)
    year = file.split("global")[1][:4]

    year_folder = OUT / year
    year_folder.mkdir(parents=True, exist_ok=True)

    n_total = len(df)
    n_preview = 0
    n_saved = 0
    n_errors = 0

    print(f"\n=== {year} | filas: {n_total} ===")

    # Spotify API permite batch de 50, pero para debug vamos 1x1 con logs
    for _, row in tqdm(df.iterrows(), total=n_total):
        tid = track_id_from_uri(row["uri"])

        try:
            t = sp.track(tid)
            preview = t.get("preview_url")

            if not preview:
                continue

            n_preview += 1

            # descarga
            r = requests.get(preview, timeout=20)
            if r.status_code != 200 or not r.content:
                n_errors += 1
                continue

            # nombre seguro
            fname = year_folder / f"{tid}_{safe_name(row.get('track_name','track'))}.mp3"
            with open(fname, "wb") as f:
                f.write(r.content)

            n_saved += 1

            # para no spamear Spotify
            time.sleep(0.05)

        except Exception as e:
            n_errors += 1
            # imprime solo algunos errores para no llenar pantalla
            if n_errors <= 5:
                print(f"\n[ERR] {tid}: {type(e).__name__}: {e}")
            continue

    print(f"Previews disponibles: {n_preview}")
    print(f"Guardados: {n_saved}")
    print(f"Errores: {n_errors}")
    print(f"Carpeta: {year_folder.resolve()}")


=== 2017 | filas: 200 ===


100%|██████████| 200/200 [00:33<00:00,  6.02it/s]


Previews disponibles: 0
Guardados: 0
Errores: 0
Carpeta: /Users/emiliahernandez/Desktop/desarrollo/Proyetco/sounds/previews/2017

=== 2021 | filas: 200 ===


100%|██████████| 200/200 [00:34<00:00,  5.78it/s]


Previews disponibles: 0
Guardados: 0
Errores: 0
Carpeta: /Users/emiliahernandez/Desktop/desarrollo/Proyetco/sounds/previews/2021

=== 2025 | filas: 200 ===


100%|██████████| 200/200 [00:33<00:00,  5.90it/s]

Previews disponibles: 0
Guardados: 0
Errores: 0
Carpeta: /Users/emiliahernandez/Desktop/desarrollo/Proyetco/sounds/previews/2025


In [4]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from tqdm import tqdm
from pathlib import Path
import time

CLIENT_ID = "2a1f45380fda4f0ba4661f6e44882f59"
CLIENT_SECRET = "024f5dc449d04b27aaf01f4da41c95ff"

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
)

files = {
    2017: "topGlobal/junio_primera_semana/global2017_first_week_june_2017-06-01.csv",
    2021: "topGlobal/junio_primera_semana/global2021_first_week_june_2021-06-03.csv",
    2025: "topGlobal/junio_primera_semana/global2025_first_week_june_2025-06-05.csv",
}

def track_id_from_uri(uri: str) -> str:
    return str(uri).split(":")[-1]

def chunks(lst, n=50):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

all_years = []

for year, fp in files.items():
    df = pd.read_csv(fp).copy()
    df["track_id"] = df["uri"].apply(track_id_from_uri)
    df["year"] = year

    # IDs únicos (evita repetir)
    ids = df["track_id"].dropna().unique().tolist()

    feats = []
    for batch in tqdm(list(chunks(ids, 50)), desc=f"Audio features {year}"):
        try:
            res = sp.audio_features(batch)  # lista de dicts (o None)
            feats.extend([r for r in res if r is not None])
            time.sleep(0.05)
        except Exception as e:
            print(f"[WARN] batch falló {year}: {e}")
            continue

    feats_df = pd.DataFrame(feats)
    feats_df["year"] = year

    # merge: metadata (rank/streams/etc) + features
    out = df.merge(feats_df, on=["track_id", "year"], how="left")
    all_years.append(out)

final = pd.concat(all_years, ignore_index=True)

# columnas útiles (puedes ajustar)
keep = [
    "year","rank","track_id","uri","artist_names","track_name","streams","source_file",
    "danceability","energy","speechiness","acousticness","instrumentalness",
    "liveness","valence","tempo","loudness","key","mode","time_signature","duration_ms"
]
final = final[[c for c in keep if c in final.columns]]

Path("sounds").mkdir(exist_ok=True)
final.to_csv("sounds/audio_features_june_first_week_2017_2021_2025.csv", index=False)

print("Listo -> sounds/audio_features_june_first_week_2017_2021_2025.csv")
print("Coverage (tracks con features):", final["danceability"].notna().mean())

Audio features 2017:  25%|██▌       | 1/4 [00:00<00:00,  6.62it/s]HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=3umS4y3uQDkqekNjVpiRUs,3NdDpSvN911VPGivFlV5d0,3kxfsdsCpFgN412fpnW85Y,2kVHAQKVtczchKctctzbtK,5GXAXm5YOmYT0kL5jHvYBt,2mfUa8bLs2s5N4VaqJZ4lZ,5uCax9HTNlzGybIStD3vDh,00lNx0OcTJrS3MKHcB80HY,7abpmGpF7PGep2rDU68GBR,6HUnnBwYZqcED1eQztxMBN,7BKLCZ1jbUBVqRi2FVlTVw,2Vdub5mY4lad7w64bFPUez,6PGoSes0D9eUDeeAafB2As,6gBFPUFcJLzWGx4lenP6h2,2bjwRfXMk4uRgOD9IBYl9h,209gZgcfLq2aUuu51vOWBl,61WbtB6ujkpNAsAf5LjF4b,78rIJddV4X0HkNAInEcYde,6mORGLOz79w6VsCRLWYYuK,5aAx2yezTd8zXrkmtKl66Z,4Q3N4Ct4zCuIHuZ65E3BD4,6PCUP3dWmTjcTtXY02oFdT,6dVZCbi9CYtD9LjAHXRjIG,4c2W3VKsOFoIg2SFaO6DY5,6mICuAdrwEjh6Y6lroV2Kg,6qDF4wWL49CAVbgT7yuHl8,1xznGGDReH1oQq0xzbwXa3,6SwRhMLwNqEi6alNPVG00n,000xQL6tZNLJzIrtIgxqSl,7mldq42yDuxiUNn08nvzHO,6XOYVSmNDjKUNMXooU4s4z,5NNlUMcOEOdoOIwwaWXv0k,4Q5yMlwAfAoitqg4r9oZHN,3bWAqKDWg6u1davspr5IkS,4Km5HrUvYTaSUfiSGPJeQR,4qknM1pQz53QOyfDVTjcM9,6p8NuHm8uCGnn2Dtbtf7zE,2Ce5IyMlVRVvN9

[WARN] batch falló 2017: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=5CtI0qwDJkDQGwXD1H1cLb,3DXncPQOG4VBw3QHh3S817,7qiZfU4dY1lWllzX7mPBI3,7KXjTSCq5nL1LoYtL7XAwS,4aWmUDTfIPGksMNLV2rQP2,6kex4EBAj0WHXDKZMEJaaF,79cuOz3SPQTuFrp8WgftAu,4iLqG9SeJSnt0cSPICSjxv,0dA2Mk56wEzDgegdC6R17g,3eR23VReFzcdmS7TYCrhCe,2habSXqcJGExM6JJyskY7O,6RUKPb4LETWmmr3iAEQktW,0KKkJNfGyhkQ5aFogxQAPU,3B54sVLJ402zGa6Xm4YGNe,3rOSwuTsUlJp0Pu0MkN8r8,1x5sYLZiu9r5E43kMlt9f8,2eMwDehkIC1j68U6FA3Eiq,0qYTZCo5Bwh1nsUFGZP3zn,0tKcYR2II1VCQWT79i5NrW,6EpRaXYhGOB3fj4V2uDkMJ,1sCxVKWImDZSZKvG0U9B23,2Gl0FzuLxflY6nPifJp5Dr,5Ohxk2dO5COHF1krpoPigN,1NDxZ7cFAo481dtYWdrUnR,1louJpMmzEicAn7lzDalPW,7hDc8b7IXETo14hHIHdnhd,6jA8HL9i4QGzsj6fjoxp8Y,6D0b04NJIKfEMg040WioJQ,3a1lNhkSLSkpJE4MSHpDu9,7tr2za8SQg2CI8EDgrdtNl,0afhq8XCExXpqazXczTSve,7nKBxz47S9SD79N086fuhn,5tz69p7tJuGPeMGwNTxYuV,0NiXXAI876aGImAd6rTj8w,6De0lHrwBfPfrhorm9q1Xl,6HZILIRieu8S0iqY8kIKhj,3ebXMykcMXOcLeJ9xZ17XH,167NczpNbRF7oWakJaY3Hh,7i2DJ88J7jQ8K7zqFX2fW8,51

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=4IWAyPf1KMq7JCyGeCjTeH,6520aj0B4FSKGVuKNsOCOi,1e1JKLEDKP7hEQzJfNAgPl,0wfbD5rAksdXUzRvMfM3x5,1qOLh0tI7trd1zdDKxYZTe,5q2JbCNi4FcnglgPfxcV65,5Ua3GXyHwiSfpNTMjq6m2z,7KOlJ92bu51cltsD9KU5I7,5q5gzmbBS5yQzos2BvVr1t,3EfugazgSddQvzZpkof5I4,3E2Zh20GDCR9B1EYjfXWyv,3AEZUABDXNtecAOSC1qTfo,11EDhDAVDtGPoSar6ootYA,3QWjljChcOMkRDYSzF33Qr,0QsvXIfqM0zZoerQfsI9lm,0gbBzIqrECJOEPvQJIBFs5,3b5Li4QKDVBx1x7fQuu54a,1EaKU4dMbesXXd3BrLCtYG,5hEM0JchdVzQ5PwvSfITeX,5kRPPEWFJIMox5qIkQkiz5,4pLwZjInHj3SimIyN9SnOz,6mapJIPnQ23RTAevUoE0DL,04DwTuZ2VBdJCCC5TROn7L,6fxVffaTuwjgEk5h9QyRjy,1a5Yu5L18qNxVhXx38njON,1yxgsra98r3qAtxqiGZPiX,0FuTx2s3YH1ppmtiM6l0zI,6JV2JOEocMgcZxYSZelKcc,7DM4BPaS7uofFul3ywMe46,68EMU2RD1ECNeOeJ5qAXCV,4Hf7WnR761jpxPr5D46Bcd,2RttW7RAu5nOAfq6YFvApB,17Fd6Yb7mSbinKG8LoWfFl,6DNtNfH8hXkqOX1sjqmI7p,0fg8CqpjdojMyXLNzM2PaJ,5GG3knKdxKWrNboRijxeKF,5MFzQMkrl1FOOng9tq6R9r,4gqMQftUs22F8pOGVn5Acr,1Slwb6dOYkBlWal1PGtnNg,0pSBuHjILhNEo55xK1zrRt,7lRsNbdOGyk

[WARN] batch falló 2017: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=4IWAyPf1KMq7JCyGeCjTeH,6520aj0B4FSKGVuKNsOCOi,1e1JKLEDKP7hEQzJfNAgPl,0wfbD5rAksdXUzRvMfM3x5,1qOLh0tI7trd1zdDKxYZTe,5q2JbCNi4FcnglgPfxcV65,5Ua3GXyHwiSfpNTMjq6m2z,7KOlJ92bu51cltsD9KU5I7,5q5gzmbBS5yQzos2BvVr1t,3EfugazgSddQvzZpkof5I4,3E2Zh20GDCR9B1EYjfXWyv,3AEZUABDXNtecAOSC1qTfo,11EDhDAVDtGPoSar6ootYA,3QWjljChcOMkRDYSzF33Qr,0QsvXIfqM0zZoerQfsI9lm,0gbBzIqrECJOEPvQJIBFs5,3b5Li4QKDVBx1x7fQuu54a,1EaKU4dMbesXXd3BrLCtYG,5hEM0JchdVzQ5PwvSfITeX,5kRPPEWFJIMox5qIkQkiz5,4pLwZjInHj3SimIyN9SnOz,6mapJIPnQ23RTAevUoE0DL,04DwTuZ2VBdJCCC5TROn7L,6fxVffaTuwjgEk5h9QyRjy,1a5Yu5L18qNxVhXx38njON,1yxgsra98r3qAtxqiGZPiX,0FuTx2s3YH1ppmtiM6l0zI,6JV2JOEocMgcZxYSZelKcc,7DM4BPaS7uofFul3ywMe46,68EMU2RD1ECNeOeJ5qAXCV,4Hf7WnR761jpxPr5D46Bcd,2RttW7RAu5nOAfq6YFvApB,17Fd6Yb7mSbinKG8LoWfFl,6DNtNfH8hXkqOX1sjqmI7p,0fg8CqpjdojMyXLNzM2PaJ,5GG3knKdxKWrNboRijxeKF,5MFzQMkrl1FOOng9tq6R9r,4gqMQftUs22F8pOGVn5Acr,1Slwb6dOYkBlWal1PGtnNg,0p

KeyError: 'track_id'

In [6]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

CLIENT_ID = "2a1f45380fda4f0ba4661f6e44882f59"
CLIENT_SECRET = "ff71c16f059e425ba0741e5acb3bbb2b"

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
))

print(sp.audio_features(["6RUKPb4LETWmmr3iAEQktW"])[0])  # Closer (debería regresar dict)

HTTP Error for GET to https://api.spotify.com/v1/audio-features/?ids=6RUKPb4LETWmmr3iAEQktW with Params: {} returned 403 due to None


SpotifyException: http status: 403, code: -1 - https://api.spotify.com/v1/audio-features/?ids=6RUKPb4LETWmmr3iAEQktW:
 None, reason: None

In [7]:
print(sp.track("6RUKPb4LETWmmr3iAEQktW")["name"])

Something Just Like This


In [8]:
import pandas as pd
import requests
from pathlib import Path
from urllib.parse import quote
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from requests.exceptions import HTTPError, RequestException, Timeout
from urllib3.util.retry import Retry
import re
import time

files = {
    2017: "topGlobal/junio_primera_semana/global2017_first_week_june_2017-06-01.csv",
    2021: "topGlobal/junio_primera_semana/global2021_first_week_june_2021-06-03.csv",
    2025: "topGlobal/junio_primera_semana/global2025_first_week_june_2025-06-05.csv",
}

OUT = Path("sounds/apple_previews")
OUT.mkdir(parents=True, exist_ok=True)

def safe_name(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-\.]+", "_", s)
    return s[:120]

def itunes_search(track, artist, country="US", limit=3):
    q = quote(f"{track} {artist}")
    url = f"https://itunes.apple.com/search?term={q}&entity=song&country={country}&limit={limit}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.json().get("results", [])

for year, fp in files.items():
    df = pd.read_csv(fp).copy()
    year_dir = OUT / str(year)
    year_dir.mkdir(exist_ok=True)

    saved = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Apple previews {year}"):
        track = str(row["track_name"])
        artist = str(row["artist_names"]).split(",")[0]  # primer artista para buscar mejor

        try:
            results = itunes_search(track, artist, country="US", limit=3)
            if not results:
                continue

            # toma el primer match
            preview = results[0].get("previewUrl")
            if not preview:
                continue

            audio = requests.get(preview, timeout=30)
            if audio.status_code != 200 or not audio.content:
                continue

            fname = year_dir / f"{safe_name(track)}__{safe_name(artist)}.m4a"
            with open(fname, "wb") as f:
                f.write(audio.content)

            saved += 1

        except Exception:
            continue

    print(f"{year}: guardados {saved} previews en {year_dir}")

Apple previews 2017: 100%|██████████| 200/200 [03:59<00:00,  1.20s/it]


2017: guardados 161 previews en sounds/apple_previews/2017


Apple previews 2021: 100%|██████████| 200/200 [04:25<00:00,  1.33s/it]


2021: guardados 180 previews en sounds/apple_previews/2021


Apple previews 2025: 100%|██████████| 200/200 [04:10<00:00,  1.25s/it]

2025: guardados 193 previews en sounds/apple_previews/2025


In [4]:
# Descargar previews de todas las canciones del Top 200 en abril, mayo y septiembre
# para 2017, 2021 y 2025, guardando en las carpetas anuales ya existentes.

import pandas as pd
import requests
from pathlib import Path
from urllib.parse import quote
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from requests.exceptions import HTTPError, RequestException, Timeout
from urllib3.util.retry import Retry
import re
import time

files = {
    2017: "topGlobal/global2017.csv",
    2021: "topGlobal/global2021.csv",
    2025: "topGlobal/global2025.csv",  # asumo que "2015" fue typo por 2025
}

months_keep = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]  
OUT = Path("sounds/apple_previews")
OUT.mkdir(parents=True, exist_ok=True)

def safe_name(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^\w\-\.]+", "_", s)
    return s[:120]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
})

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
    respect_retry_after_header=True,
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

def itunes_search(track, artist, country="US", limit=5):
    q = quote(f"{track} {artist}")
    url = f"https://itunes.apple.com/search?term={q}&entity=song&country={country}&limit={limit}"
    r = session.get(url, timeout=(10, 30))
    r.raise_for_status()
    return r.json().get("results", [])

def download_preview(preview_url):
    r = session.get(preview_url, timeout=(10, 30), stream=True)
    if r.status_code in (403, 404):
        return None
    r.raise_for_status()
    content = r.content
    return content if content else None

for year, fp in files.items():
    df_year = pd.read_csv(fp).copy()
    df_year["date"] = pd.to_datetime(
        df_year["source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})", expand=False)
    )
    df_year["rank"] = pd.to_numeric(df_year["rank"], errors="coerce")

    subset = df_year[
        df_year["rank"].between(1, 200)
        & df_year["date"].dt.month.isin(months_keep)
    ].copy()

    subset = subset.drop_duplicates(subset=["uri"])
    subset = subset.sort_values(["date", "rank"]).reset_index(drop=True)

    year_dir = OUT / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    saved = 0
    skipped_existing = 0
    no_preview = 0
    errors = 0
    rate_limited = 0
    timed_out = 0

    print(f"\n=== {year} ===")
    print(f"Canciones únicas dentro del Top 200 en cada año: {len(subset)}")

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Apple previews {year}"):
        track = str(row["track_name"])
        artist = str(row["artist_names"]).split(",")[0].strip()
        fname = year_dir / f"{safe_name(track)}__{safe_name(artist)}.m4a"

        if fname.exists():
            skipped_existing += 1
            continue

        try:
            results = itunes_search(track, artist, country="US", limit=5)
            if not results:
                no_preview += 1
                continue

            preview = None
            for item in results:
                candidate = item.get("previewUrl")
                if candidate:
                    preview = candidate
                    break

            if not preview:
                no_preview += 1
                continue

            audio = download_preview(preview)
            if not audio:
                no_preview += 1
                continue

            with open(fname, "wb") as f:
                f.write(audio)

            saved += 1
            time.sleep(0.15)

        except Timeout:
            timed_out += 1
            errors += 1
            time.sleep(1.5)
            continue
        except HTTPError as e:
            status = e.response.status_code if e.response is not None else None
            if status == 429:
                rate_limited += 1
                time.sleep(3)
            elif status in (403, 404):
                no_preview += 1
                continue
            errors += 1
            continue
        except RequestException:
            errors += 1
            time.sleep(1.0)
            continue

    print(f"{year}: guardados {saved} nuevos previews en {year_dir}")
    print(f"{year}: ya existían {skipped_existing}")
    print(f"{year}: sin preview/match {no_preview}")
    print(f"{year}: errores {errors}")
    print(f"{year}: rate limited {rate_limited}")
    print(f"{year}: timeouts {timed_out}")



=== 2017 ===
Canciones únicas dentro del Top 200 en cada año: 958


Apple previews 2017: 100%|██████████| 958/958 [05:57<00:00,  2.68it/s] 


2017: guardados 0 nuevos previews en sounds/apple_previews/2017
2017: ya existían 715
2017: sin preview/match 243
2017: errores 0
2017: rate limited 0
2017: timeouts 0

=== 2021 ===
Canciones únicas dentro del Top 200 en cada año: 1162


Apple previews 2021: 100%|██████████| 1162/1162 [17:00<00:00,  1.14it/s] 


2021: guardados 0 nuevos previews en sounds/apple_previews/2021
2021: ya existían 588
2021: sin preview/match 574
2021: errores 0
2021: rate limited 0
2021: timeouts 0

=== 2025 ===
Canciones únicas dentro del Top 200 en cada año: 979


Apple previews 2025: 100%|██████████| 979/979 [15:58<00:00,  1.02it/s]  

2025: guardados 0 nuevos previews en sounds/apple_previews/2025
2025: ya existían 417
2025: sin preview/match 562
2025: errores 0
2025: rate limited 0
2025: timeouts 0
